# 00 - Contexto, problema de negócio


## Problema de negócio

A Olist precisa entender quais fatores afetam desempenho comercial, atraso logístico e satisfação dos clientes no marketplace.

Pergunta central:

> Quais fatores explicam pedidos com pior experiência e como priorizar ações para reduzir reviews baixos?

Hipóteses iniciais:

- H1: pedidos entregues com atraso tem maior chance de review baixo.
- H2: frete alto em relação ao valor dos produtos prejudica a avaliação.
- H3: categorias e estados específicos concentram atrasos e insatisfação.
- H4: pedidos com maior complexidade, como muitos itens ou vendedores, tem pior experiência.
- H5: um modelo simples consegue ranquear pedidos por risco de review baixo melhor do que uma regra manual.

## Plano dos notebooks

1. Entendimento dos arquivos.
2. EDA, qualidade dos dados e integração das tabelas.
3. Teste das hipóteses de negócio.
4. Feature engineering e dataset de modelagem.
5. Modelo de risco de review baixo.
6. Insights executivos e recomendações.

In [ ]:
from pathlib import Path
import sys
import warnings

import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from olist.config import RAW_DATA_DIR, ensure_project_dirs
from olist.data import CSV_FILES, load_table, validate_raw_files

ensure_project_dirs()

print(f'Project root: {PROJECT_ROOT}')
print(f'Raw data dir: {RAW_DATA_DIR}')

In [ ]:
file_status = validate_raw_files()
display(file_status)

missing_required = file_status.loc[file_status['required'] & ~file_status['exists'], 'file_name'].tolist()
assert not missing_required, f'Arquivos obrigatorios ausentes: {missing_required}'

## Amostra das tabelas


In [ ]:
for table_name, row in file_status.set_index('table').iterrows():
    if not row['exists']:
        continue
    sample = load_table(table_name, nrows=5)
    print('\n' + '=' * 80)
    print(table_name, sample.shape)
    display(sample.head())

## Decisoes iniciais

- A unidade principal de analise será o pedido (`order_id`).
- O alvo de satisfação sera `review_score <= 2`, interpretado como review baixo.
- Pedidos nao entregues serão analisados na EDA, mas a modelagem de satisfação usará apenas pedidos entregues com review.
- A tabela de geolocalização e opcional nesta versão do dashboard.